In [0]:
from pyspark.ml.regression import LinearRegression
from pyspark.ml.linalg import Vectors, VectorUDT
from pyspark.sql.functions import col

# Load sample data
data = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("/databricks-datasets/samples/population-vs-price/data_geo.csv")

# Prepare data for ML
data = data.dropna()
exprs = [col(column).alias(column.replace(' ', '_')) for column in data.columns]
spark.udf.register("oneElementVec", lambda d: Vectors.dense([d]), returnType=VectorUDT())
tdata = data.select(*exprs).selectExpr("oneElementVec(2014_Population_estimate) as features", "2015_median_sales_price as label")

# Define and fit the model
lr = LinearRegression()
model = lr.fit(tdata)

# Make predictions
predictions = model.transform(tdata)
display(predictions)
